# Stream KG Creation to Neo4j (for large graphs)

For big corpora the full knowledge graph may not fit in RAM, so instead of
building an in-memory `networkx.DiGraph` and uploading it afterwards, we
**stream nodes and edges directly into Neo4j** as they are produced.

The build logic itself is **storage-agnostic**: the same `build_kg_into`
routine writes through any `GraphWriter` backend — an in-memory
`NetworkXGraphWriter` (python object) or `Neo4jGraphBuilder` (Neo4j stream).
Swap the writer, keep the pipeline code.

- `StreamingPipeline` never materialises the whole graph — it writes to Neo4j
  via `Neo4jGraphBuilder` (one `MERGE` per node / edge).
- `execute_batched(batch_size=N)` streams documents from disk in fixed-size
  batches, so memory stays bounded no matter how large the corpus is.
- Export streams the graph *out* of Neo4j to JSON — again without loading it
  all into RAM.

**Prerequisites:** a running Neo4j instance and `NEO4J_URI`, `NEO4J_USER`,
`NEO4J_PASSWORD` set in the environment (or `.env`).

In [ ]:
from polygraph._shared import Document, Ontology
from polygraph.kg_build import build_kg_into, extract
from polygraph.kg_export.neo4j.builder import Neo4jGraphBuilder
from polygraph.pipelines import StreamingPipeline
from polygraph.preprocess import chunk, clean, load, quality


class Neo4jStreamingPipeline(StreamingPipeline):
    """Build the KG directly inside Neo4j — the full graph never sits in RAM."""

    def __init__(self, ontology: Ontology, **kwargs):
        self.ontology = ontology
        super().__init__(**kwargs)

    def preprocess(self) -> list[Document]:
        docs = clean.normalize(load.from_paths(self.input_paths))
        chunks = chunk.by_sentence(docs, target_tokens=450, overlap_tokens=60)
        return quality.filter(chunks)

    def build_kg_streaming(self, chunks: list[Document], builder: Neo4jGraphBuilder) -> None:
        # Extract entities + triples (only this batch lives in RAM), then
        # stream them into Neo4j via the shared, storage-agnostic routine.
        entities, triples = extract.with_methods(chunks, self.ontology)
        build_kg_into(builder, chunks, entities, triples)

In [ ]:
# THE key idea: one build routine, any storage. Run the SAME build logic
# against a plain Python object first — no Neo4j required.
from polygraph.kg_build import NetworkXGraphWriter

ontology = Ontology.from_yaml("configs/default_ontology.yaml")

sample = clean.normalize(load.from_paths(["data/wikipedia/connected.jsonl"]))[:3]
sample_chunks = quality.filter(chunk.by_sentence(sample, target_tokens=450, overlap_tokens=60))
entities, triples = extract.with_methods(sample_chunks, ontology)

writer = NetworkXGraphWriter(ontology=ontology)
build_kg_into(writer, sample_chunks, entities, triples)
graph = writer.graph  # an nx.DiGraph (python object)

print(f"In-memory graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
# Now run the FULL corpus with the same build routine, streaming it into
# Neo4j instead of RAM. clear_db=True wipes Neo4j first — set to False to
# append onto an existing graph instead.
pipe = Neo4jStreamingPipeline(
    ontology=ontology,
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/neo4j_streaming/",
    clear_db=True,
)

pipe.execute_batched(batch_size=50)

In [ ]:
# The graph now lives in Neo4j — verify it with a direct Cypher query.
import os

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
    auth=(
        os.environ.get("NEO4J_USER", "neo4j"),
        os.environ.get("NEO4J_PASSWORD", ""),
    ),
)

with driver.session() as session:
    nodes = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    rels = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    print(f"Neo4j now holds {nodes} nodes and {rels} relationships")

driver.close()

## Notes for very large graphs

- The build logic is **storage-agnostic**: `build_kg_into` writes through any
  `GraphWriter`. The exact same pipeline code targets `NetworkXGraphWriter`
  (in-memory) or `Neo4jGraphBuilder` (streamed) — just swap the writer. To
  add a new backend (e.g. SQLite), implement the `GraphWriter` interface and
  reuse the same routine.
- `execute_batched` deliberately skips **cross-batch** dedup and entity
  resolution (each batch is processed independently) — that is the
  memory-vs-quality trade-off that makes unbounded corpus sizes possible.
  For smaller corpora, `pipe.execute()` works the same way but preprocesses
  all documents at once.
- Export runs automatically at the end of `execute_batched` — it streams the
  graph out of Neo4j into `output/neo4j_streaming/knowledge_graph.json`
  without loading it into RAM. Pass `skip_export=True` if you only need the
  data inside Neo4j.
- Use `builder.compute_pagerank()` (Neo4j GDS) to add graph-level importance
  scores without ever materialising the graph in Python.